<a href="https://colab.research.google.com/github/firatmio/mbpp-tr-finetune/blob/main/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mbpp-tr LoRA fine-tune (Qwen3-1.7B)

`firatmio/mbpp-tr` ile `Qwen/Qwen3-1.7B` üzerinde LoRA eğitimi ve **çalıştırılarak** ölçülen pass@1 değerlendirmesi.

Akış: harness doğrulama → baseline eval → eğitim → LoRA eval → karşılaştırma → Hub'a yükleme.

**Runtime:** `Runtime > Change runtime type > T4 GPU`

In [ ]:
!nvidia-smi

Wed Sep 16 18:44:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Tekrar calistirilabilir: her zaman /content altina tek kopya, varsa sadece gunceller
%cd /content
![ -d mbpp-tr-finetune ] || git clone https://github.com/firatmio/mbpp-tr-finetune.git
%cd /content/mbpp-tr-finetune
!git pull
!pip install -q -U -r requirements.txt
# Colab'daki eski torchao (0.10) yeni peft ile uyumsuz ve bu projede kullanilmiyor
!pip uninstall -q -y torchao

/content
Cloning into 'mbpp-tr-finetune'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 48 (delta 20), reused 39 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 20.07 KiB | 10.04 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/mbpp-tr-finetune
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 62.1 MB/s eta 0:00:00


In [4]:
!python scripts/evaluate.py --config full --split validation --out outputs/val_base

README.md: 100% 9.71k/9.71k [00:00<00:00, 4.49MB/s]
mbpp_tr_train.jsonl: 100% 257k/257k [00:00<00:00, 33.3MB/s]
mbpp_tr_test.jsonl: 100% 351k/351k [00:00<00:00, 119MB/s]
mbpp_tr_validation.jsonl: 100% 61.8k/61.8k [00:00<00:00, 72.3MB/s]
mbpp_tr_prompt.jsonl: 100% 6.78k/6.78k [00:00<00:00, 15.3MB/s]
Generating train split: 374 examples [00:00, 15972.44 examples/s]
Generating test split: 500 examples [00:00, 100174.44 examples/s]
Generating validation split: 90 examples [00:00, 30810.26 examples/s]
Generating prompt split: 10 examples [00:00, 3881.46 examples/s]
config.json: 100% 726/726 [00:00<00:00, 3.18MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 25.8MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 95.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 102MB/s]

tokenizer.json: downloading bytes:   1% 152k/11.4M [00:01<01:23, 135kB/s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:01<00:00, 2.85MB/s,  329kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.

In [5]:
from huggingface_hub import login
login()

In [6]:
!python scripts/evaluate.py --config full --split validation --adapter firatmio/qwen3-1.7b-mbpp-tr-lora --out outputs/val_exp1

Loading weights: 100% 311/311 [00:02<00:00, 136.24it/s]
adapter_config.json: 100% 1.17k/1.17k [00:00<00:00, 3.26MB/s]

adapter_model.safetensors: downloading bytes:   0% 16.9k/69.8M [00:02<3:15:33, 5.95kB/s]
adapter_model.safetensors: downloading bytes:  93% 65.0M/69.8M [00:07<00:00, 16.6MB/s, 4.75MB/s  ]
adapter_model.safetensors: downloading bytes: 100% 65.3M/65.3M [00:07<00:00, 8.23MB/s, 4.85MB/s  ]
adapter_model.safetensors: reconstructing file: 100% 69.8M/69.8M [00:07<00:00, 8.79MB/s, 5.88MB/s  ]
  16/90
  32/90
  48/90
  64/90
  80/90
  90/90
{
  "model_id": "Qwen/Qwen3-1.7B",
  "adapter": "firatmio/qwen3-1.7b-mbpp-tr-lora",
  "dataset": "firatmio/mbpp-tr/full/validation",
  "n": 90,
  "passed": 37,
  "pass@1": 0.4111,
  "status_counts": {
    "error": 9,
    "assertion_error": 44,
    "passed": 37
  },
  "decoding": {
    "do_sample": false,
    "max_new_tokens": 1024
  }
}


In [7]:
!git pull
!python scripts/generate_rft_data.py --out outputs/rft

remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.61 KiB | 1.81 MiB/s, done.
From https://github.com/firatmio/mbpp-tr-finetune
   ee79ed1..46350a1  main       -> origin/main
Updating ee79ed1..46350a1
Fast-forward
 scripts/evaluate.py          |  31 ++++++-----
 scripts/generate_rft_data.py | 125 +++++++++++++++++++++++++++++++++++++++++++
 scripts/train_lora.py        |  30 +++++++++--
 3 files changed, 169 insertions(+), 17 deletions(-)
 create mode 100644 scripts/generate_rft_data.py
Loading weights: 100% 311/311 [00:01<00:00, 237.02it/s]
Tur 0 (greedy): 374 gorev
  16/374
  32/374
  48/374
  64/374
  80/374
  96/374
  112/374
  128/374
  144/374
  160/374
  176/374
  192/374
  208/374
  224/374
  240/374
  256/374
  272/374
  288/374
  304/374
  320/374
  336/374
  352/374
  368/374
  374/374
  {'asse

In [8]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/mbpp-tr-finetune/rft
!cp outputs/rft/* /content/drive/MyDrive/mbpp-tr-finetune/rft/

Mounted at /content/drive


In [9]:
!python scripts/train_lora.py --output_dir outputs/lora_rft --rft_file outputs/rft/train.jsonl --epochs 2 --lr 1e-4

mbpp_tr_train.jsonl: 100% 90.7k/90.7k [00:00<00:00, 26.2MB/s]
mbpp_tr_test.jsonl: 100% 185k/185k [00:00<00:00, 23.5MB/s]
mbpp_tr_validation.jsonl: 100% 28.5k/28.5k [00:00<00:00, 37.1MB/s]
mbpp_tr_prompt.jsonl: 100% 4.73k/4.73k [00:00<00:00, 4.97MB/s]
Generating train split: 120 examples [00:00, 20894.04 examples/s]
Generating test split: 257 examples [00:00, 76416.85 examples/s]
Generating validation split: 43 examples [00:00, 16235.04 examples/s]
Generating prompt split: 7 examples [00:00, 2572.74 examples/s]
Filter: 100% 374/374 [00:00<00:00, 37204.01 examples/s]
Map: 100% 151/151 [00:00<00:00, 7468.54 examples/s]
Map: 100% 151/151 [00:00<00:00, 489.06 examples/s]
Map: 100% 90/90 [00:00<00:00, 967.74 examples/s]
train=151 val=90 max_tokens=585
Loading weights: 100% 311/311 [00:01<00:00, 220.52it/s]
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030
{'loss': '0.09033', 'grad_norm': '1.045', 'learning_rate': '9.397e-05', 'epoch': '0.5263'}
{'loss': '0.0872'

In [10]:
!python scripts/evaluate.py --adapter outputs/lora_rft/final --out outputs/eval_rft

Loading weights: 100% 311/311 [00:01<00:00, 225.16it/s]
  16/257
  32/257
  48/257
  64/257
  80/257
  96/257
  112/257
  128/257
  144/257
  160/257
  176/257
  192/257
  208/257
  224/257
  240/257
  256/257
  257/257
{
  "model_id": "Qwen/Qwen3-1.7B",
  "adapter": "outputs/lora_rft/final",
  "dataset": "firatmio/mbpp-tr/sanitized/test",
  "n": 257,
  "passed": 141,
  "pass@1": 0.5486,
  "status_counts": {
    "passed": 141,
    "assertion_error": 81,
    "error": 35
  },
  "decoding": {
    "do_sample": false,
    "max_new_tokens": 1024
  }
}


In [11]:
!python scripts/evaluate.py --config full --split validation --adapter outputs/lora_rft/final --out outputs/val_rft

Loading weights: 100% 311/311 [00:01<00:00, 225.46it/s]
  16/90
  32/90
  48/90
  64/90
  80/90
  90/90
{
  "model_id": "Qwen/Qwen3-1.7B",
  "adapter": "outputs/lora_rft/final",
  "dataset": "firatmio/mbpp-tr/full/validation",
  "n": 90,
  "passed": 31,
  "pass@1": 0.3444,
  "status_counts": {
    "assertion_error": 38,
    "error": 21,
    "passed": 31
  },
  "decoding": {
    "do_sample": false,
    "max_new_tokens": 1024
  }
}


In [12]:
!mkdir -p /content/drive/MyDrive/mbpp-tr-finetune/exp2
!cp -r outputs/lora_rft/final outputs/lora_rft/train_summary.json outputs/eval_rft outputs/val_rft outputs/val_base outputs/val_exp1 /content/drive/MyDrive/mbpp-tr-finetune/exp2/

## 1. Harness doğrulama
Referans çözümler kendi testlerini geçmiyorsa, model sonuçları da güvenilir değildir. Beklenen: `257/257`.

In [ ]:
!python scripts/check_references.py --config sanitized --split test

## 2. Baseline (fine-tune öncesi)
Aynı prompt, greedy decoding. T4'te ~10-20 dk sürebilir.

In [ ]:
!python scripts/evaluate.py --out outputs/eval_base

## 3. LoRA eğitimi
Varsayılanlar: r=16, alpha=32, dropout=0.05, lr=2e-4, 3 epoch, efektif batch 16, cosine. Her epoch sonunda validation loss; en iyisi saklanır.

In [ ]:
!python scripts/train_lora.py --output_dir outputs/lora

## 4. LoRA sonrası değerlendirme

In [ ]:
!python scripts/evaluate.py --adapter outputs/lora/final --out outputs/eval_lora

## 5. Karşılaştırma

In [ ]:
import json
rows = {}
for name in ["base", "lora"]:
    with open(f"outputs/eval_{name}/summary.json") as f:
        rows[name] = json.load(f)
for name, s in rows.items():
    print(f"{name:5s} pass@1 = {s['pass@1']:.4f}  ({s['passed']}/{s['n']})  {s['status_counts']}")

# Hangi görevler kazanıldı / kaybedildi?
def load(name):
    with open(f"outputs/eval_{name}/samples.jsonl", encoding="utf-8") as f:
        return {r["task_id"]: r for r in map(json.loads, f)}
base, lora = load("base"), load("lora")
gained = [t for t in base if lora[t]["passed"] and not base[t]["passed"]]
lost = [t for t in base if base[t]["passed"] and not lora[t]["passed"]]
print(f"kazanilan: {len(gained)}  kaybedilen: {len(lost)}")
print("kaybedilenler:", lost)

## 6. Çıktıları sakla ve Hub'a yükle
Colab oturumu kapanınca `outputs/` silinir. Önce Drive'a yedekle, sonra adapter'ı Hub'a yükle (HF token'ı `write` yetkili olmalı).

In [13]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/mbpp-tr-finetune
!cp -r outputs/eval_base outputs/eval_lora outputs/lora/final outputs/lora/train_summary.json /content/drive/MyDrive/mbpp-tr-finetune/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat 'outputs/eval_base': No such file or directory
cp: cannot stat 'outputs/eval_lora': No such file or directory
cp: cannot stat 'outputs/lora/final': No such file or directory
cp: cannot stat 'outputs/lora/train_summary.json': No such file or directory


In [ ]:
from huggingface_hub import login
login()  # token'ı etkileşimli gir; notebook'a yazma

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "firatmio/qwen3-1.7b-mbpp-tr-lora"  # model card hazır olmadan private tutmak mantıklı
api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=True, exist_ok=True)
api.upload_folder(repo_id=REPO_ID, folder_path="outputs/lora/final", commit_message="LoRA adapter")
for name in ["base", "lora"]:
    api.upload_folder(repo_id=REPO_ID, folder_path=f"outputs/eval_{name}", path_in_repo=f"eval/{name}")
api.upload_file(repo_id=REPO_ID, path_or_fileobj="outputs/lora/train_summary.json", path_in_repo="train_summary.json")